<a href="https://colab.research.google.com/github/HelloSamved/learning-neural-network/blob/master/mnist_prediction/mnist_prediction_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as o
from torch.utils.data import DataLoader
print("modules imported successfully")

modules imported successfully


In [14]:
data_train= datasets.MNIST(
    root="data",
    train= True,
    transform=ToTensor(),
    download= True
)

data_test= datasets.MNIST(
    root="data",
    train= False,
    transform=ToTensor(),
    download= True
)

In [15]:
data_train

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [16]:
 data_test

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [17]:
data_train.targets

tensor([5, 0, 4,  ..., 5, 6, 8])

In [18]:
loader= {
    'train': DataLoader(data_train,
                        batch_size= 200,
                        shuffle= True,
                        num_workers=1),
    'test': DataLoader(data_test,
                        batch_size= 100,
                        shuffle= True,
                        num_workers=1)
}

In [19]:
loader

{'train': <torch.utils.data.dataloader.DataLoader at 0x7f48db384bc0>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f48daa438c0>}

In [20]:
class network(nn.Module):
  def __init__(self):
    super(network, self).__init__()
    self.conv1= nn.Conv2d(1,10,kernel_size=5)
    self.conv2= nn.Conv2d(10,20, kernel_size=5)
    self.conv2_drop= nn.Dropout2d()
    self.fc1= nn.Linear(320,50)
    self.fc2= nn.Linear(50,10)

  def forward(self,x):
    x= f.relu(f.max_pool2d(self.conv1(x),2))
    x= f.relu(f.max_pool2d(self.conv2_drop(self.conv2(x)),2))
    x= x.view(-1,320)
    x= f.relu(self.fc1(x))
    x= f.dropout(x, training= self.training)
    x= self.fc2(x)
    return f.softmax(x)

In [21]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= network().to(device)
optimizer= o.Adam(model.parameters(), lr= 0.002)
loss_function= nn.CrossEntropyLoss()

def train(epoch):
  model.train()
  for batch_idx, (data, target) in enumerate(loader['train']):
    data,target= data.to(device), target.to(device)
    optimizer.zero_grad()
    output= model(data)

    loss= loss_function(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % 30 ==0:
      print(f'Train epoch: {epoch} [{batch_idx * len(data)}/{len(loader["train"].dataset)}] Loss: {loss.item()}')

In [22]:
def test():
  model.eval()
  test_loss=0
  correct= 0

  with torch.no_grad():
    for data, target in loader['test']:
      data,target= data.to(device), target.to(device)
      output= model(data)
      test_loss+= loss_function(output,target).item()
      pred= output.argmax(dim=1, keepdim= True)
      correct+= pred.eq(target.view_as(pred)).sum().item()

  test_loss/= len(loader['test'].dataset)
  print(f'\nTest set: Average loss: {test_loss}, Accuracy: {correct}/{len(loader["test"].dataset)} ({100. * correct / len(loader["test"].dataset)} %)\n')

In [23]:
for epoch in range(1,11):
  train(epoch)
  test()

/tmp/ipykernel_4175/785357238.py:17: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return f.softmax(x)


Train epoch: 1 [0/60000] Loss: 2.303053855895996
Train epoch: 1 [3000/60000] Loss: 2.138775110244751
Train epoch: 1 [6000/60000] Loss: 1.904525637626648
Train epoch: 1 [9000/60000] Loss: 1.8239986896514893
Train epoch: 1 [12000/60000] Loss: 1.7297405004501343
Train epoch: 1 [15000/60000] Loss: 1.7432893514633179
Train epoch: 1 [18000/60000] Loss: 1.677932858467102
Train epoch: 1 [21000/60000] Loss: 1.6416428089141846
Train epoch: 1 [24000/60000] Loss: 1.6607780456542969
Train epoch: 1 [27000/60000] Loss: 1.6548036336898804
Train epoch: 1 [30000/60000] Loss: 1.6176263093948364
Train epoch: 1 [33000/60000] Loss: 1.627782940864563
Train epoch: 1 [36000/60000] Loss: 1.6332751512527466
Train epoch: 1 [39000/60000] Loss: 1.5848186016082764
Train epoch: 1 [42000/60000] Loss: 1.6293524503707886
Train epoch: 1 [45000/60000] Loss: 1.6186246871948242
Train epoch: 1 [48000/60000] Loss: 1.6360238790512085
Train epoch: 1 [51000/60000] Loss: 1.5871787071228027
Train epoch: 1 [54000/60000] Loss: 1.591